# Sprint 3 — Advanced challenger experiment (Colab launcher)

**Frozen representation:** `sentence-transformers/sentence-t5-base` — the embeddings already in
`sprint3_results/embeddings/` are reused; nothing is re-embedded and no API is called.

**Baseline:** Sentence-T5 + K-Means (configuration read from the Sprint 3 result files)

**Advanced challengers:**
- BERTopic (T5 embeddings → UMAP → HDBSCAN → c-TF-IDF topic representation)
- Leiden community detection (cosine kNN graph)
- Gaussian Mixture Model (diagonal covariance, Sprint 3 PCA space)

Principle: **complexity must earn its place** — a challenger replaces K-Means only with a meaningful gain and no
unacceptable cost. All logic lives in `sprint3_advanced_challengers.py`; this notebook only launches it and reads results.

Required in **My Drive/CAPESTONE PROJECT/**: this notebook, `sprint3_advanced_challengers.py`,
`sprint3_colab_experiment.py` (metric definitions are imported from it), `sprint3_task_input.csv`, `sprint3_results/`.
If the runtime disconnects, re-run cells 3–6: the script resumes from its checkpoints.

In [1]:
# CELL 2 — GPU / environment (the experiment itself is CPU-bound; no model is loaded)
!nvidia-smi
import torch
print("torch.cuda.is_available():", torch.cuda.is_available())

Thu Oct 22 21:36:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# CELL 3 — mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# CELL 4 — paths + required existing files
import os

PROJECT_DIR = "/content/drive/MyDrive/CAPESTONE PROJECT"
SCRIPT = f"{PROJECT_DIR}/sprint3_advanced_challengers.py"
OUTPUT = f"{PROJECT_DIR}/sprint3_advanced_results"
ZIP = f"{PROJECT_DIR}/Sprint3_Advanced_Challenger_Results.zip"
for name, path in [("SCRIPT", SCRIPT), ("OUTPUT", OUTPUT), ("ZIP", ZIP)]:
    print(f"{name:7s} {path}")

required = [
    SCRIPT,
    f"{PROJECT_DIR}/sprint3_colab_experiment.py",
    f"{PROJECT_DIR}/sprint3_task_input.csv",
    f"{PROJECT_DIR}/sprint3_results/run_manifest.json",
    f"{PROJECT_DIR}/sprint3_results/embeddings/sentence_t5_embeddings.npy",
    f"{PROJECT_DIR}/sprint3_results/embeddings/sentence_t5_embedding_mapping.csv",
    f"{PROJECT_DIR}/sprint3_results/embeddings/sentence_t5_meta.json",
    f"{PROJECT_DIR}/sprint3_results/embeddings/sentence_t5_paired_text_embeddings.npy",
    f"{PROJECT_DIR}/sprint3_results/embeddings/sentence_t5_paired_text_mapping.csv",
    f"{PROJECT_DIR}/sprint3_results/evaluation/clustering_experiments.csv",
]
missing = [p for p in required if not os.path.isfile(p)]
for p in required:
    print(("OK      " if p not in missing else "MISSING ") + p)
if missing:
    raise FileNotFoundError("Required files are missing (see above). Embeddings are NOT regenerated.")

SCRIPT  /content/drive/MyDrive/CAPESTONE PROJECT/sprint3_advanced_challengers.py
OUTPUT  /content/drive/MyDrive/CAPESTONE PROJECT/sprint3_advanced_results
ZIP     /content/drive/MyDrive/CAPESTONE PROJECT/Sprint3_Advanced_Challenger_Results.zip
OK      /content/drive/MyDrive/CAPESTONE PROJECT/sprint3_advanced_challengers.py
OK      /content/drive/MyDrive/CAPESTONE PROJECT/sprint3_colab_experiment.py
OK      /content/drive/MyDrive/CAPESTONE PROJECT/sprint3_task_input.csv
OK      /content/drive/MyDrive/CAPESTONE PROJECT/sprint3_results/run_manifest.json
OK      /content/drive/MyDrive/CAPESTONE PROJECT/sprint3_results/embeddings/sentence_t5_embeddings.npy
OK      /content/drive/MyDrive/CAPESTONE PROJECT/sprint3_results/embeddings/sentence_t5_embedding_mapping.csv
OK      /content/drive/MyDrive/CAPESTONE PROJECT/sprint3_results/embeddings/sentence_t5_meta.json
OK      /content/drive/MyDrive/CAPESTONE PROJECT/sprint3_results/embeddings/sentence_t5_paired_text_embeddings.npy
OK      /content/

In [4]:
# CELL 5 — install dependencies (BERTopic, UMAP, HDBSCAN, igraph + leidenalg, scientific stack).
# Packages Colab already provides are left as they are; no model is downloaded.
!pip install -q bertopic umap-learn hdbscan python-igraph leidenalg scikit-learn pandas numpy scipy matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 142.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 107.7 MB/s eta 0:00:00


In [5]:
# CELL 6 — run the experiment
!python "/content/drive/MyDrive/CAPESTONE PROJECT/sprint3_advanced_challengers.py" \
  --project-dir "/content/drive/MyDrive/CAPESTONE PROJECT" \
  --output "/content/drive/MyDrive/CAPESTONE PROJECT/sprint3_advanced_results"

[21:37:06 +     5s] device=cuda gpu=Tesla T4 (this experiment is CPU-bound; no model is loaded)
[21:37:11 +    10s] validated: rows=3015 postings=188 T5 embeddings (3015, 768) paired wordings=432
[21:37:12 +    10s] baseline: EXP_3_T5_KMEANS__k=100 (k=100) - Sprint 3 overall winner (run_manifest.json chosen_configuration)
[21:37:15 +    13s] common low-confidence margin (P10 of baseline primary-secondary centroid gap) = 0.0066
[21:37:26 +    24s] baseline reproduction vs Sprint 3 records: max abs diff 5.55e-17; labels ARI 1.000 -> reproduces=True
[21:37:26 +    24s] GMM n_components=60
[21:37:42 +    40s]   ADV_3_T5_GMM__n_components=60_cov=diag: comps used=60 lift=0.249 stab=0.405 BIC=-476251
[21:37:42 +    40s] GMM n_components=80
[21:37:57 +    56s]   ADV_3_T5_GMM__n_components=80_cov=diag: comps used=80 lift=0.268 stab=0.396 BIC=-466768
[21:37:57 +    56s] GMM n_components=100
[21:38:14 +    73s]   ADV_3_T5_GMM__n_components=100_cov=diag: comps used=100 lift=0.288 stab=0.377 BIC=-4

In [6]:
# CELL 7 — verify results (read-only)
from pathlib import Path

if os.path.isfile(ZIP):
    print(f"ZIP: {ZIP}\nsize: {os.path.getsize(ZIP) / 1e6:.1f} MB")
else:
    print(f"ZIP NOT FOUND: {ZIP} - check the output of cell 6.")
out = Path(OUTPUT)
print("\nResult directories:")
for d in ["bertopic", "leiden", "gmm", "baseline", "evaluation", "figures"]:
    p = out / d
    n = len([f for f in p.rglob("*") if f.is_file()]) if p.exists() else 0
    print(f"  {d:11s} {'OK ' if p.exists() else 'MISSING'} {n} files")
for f in ["run_manifest.json", "advanced_challenger_report.md"]:
    print(f"  {f:30s} {'OK' if (out / f).exists() else 'MISSING'}")

ZIP: /content/drive/MyDrive/CAPESTONE PROJECT/Sprint3_Advanced_Challenger_Results.zip
size: 0.9 MB

Result directories:
  bertopic    OK  3 files
  leiden      OK  2 files
  gmm         OK  2 files
  baseline    OK  3 files
  evaluation  OK  5 files
  figures     OK  9 files
  run_manifest.json              OK
  advanced_challenger_report.md  OK


In [7]:
# CELL 8 — results preview (reads generated files; recalculates nothing)
import json
import pandas as pd
from IPython.display import Markdown, display

cmp_path = out / "evaluation" / "advanced_challenger_comparison.csv"
if cmp_path.exists():
    cmp = pd.read_csv(cmp_path)
    display(cmp)
    base = cmp[cmp["method"] == "kmeans"].iloc[0]
    print("CURRENT BASELINE:", base["pipeline"], "|", base["parameters"])
else:
    print("advanced_challenger_comparison.csv not found")

mf = out / "run_manifest.json"
if mf.exists():
    man = json.loads(mf.read_text())
    best = man.get("best_configurations", {})
    dec = man.get("replacement_decisions", {})
    qualified = [m for m, d in dec.items() if d.get("qualifies")]
    print("BEST CHALLENGER CONFIGS:", {m: c for m, c in best.items() if m != "kmeans"})
    print("CHALLENGERS THAT QUALIFIED TO REPLACE K-MEANS:", qualified or "none")
    print("FINAL RECOMMENDATION:", man.get("final_recommendation"))
    print("KEY REASON:", "; ".join(man.get("final_trace", [])))
    for m, d in dec.items():
        print(f"  {m}: gains={d.get('gains')} costs={d.get('costs')}")

rp = out / "advanced_challenger_report.md"
if rp.exists():
    display(Markdown(rp.read_text()))

,pipeline,method,config_id,parameters,n_clusters,coverage,noise_share,semantic_cohesion,semantic_cohesion_lift,semantic_cohesion_incl_noise,...,configs_tested,method_runtime_seconds,semantic_cohesion_ci95,silhouette_ci95,paraphrase_consistency_ci95,error_analysis_rows,meaningful_gains_vs_kmeans,unacceptable_costs_vs_kmeans,replaces_kmeans,final_recommendation
0,Sentence-T5 + K-Means,kmeans,ADV_0_T5_KMEANS_BASELINE__k=100,k=100,100,1.000000,0.000000,0.914050,0.289711,0.914050,...,1.0,11.8,"[0.913, 0.915]","[0.066, 0.071]","[0.648, 0.730]",190,NaN,NaN,NaN,KEEP T5 + KMEANS
1,Sentence-T5 + BERTopic,bertopic,ADV_1_T5_BERTOPIC__nn=30_nc=5_mcs=10_ms=5,"n_neighbors=30,n_components=5,min_dist=0.0,met...",70,0.727695,0.272305,0.914586,0.292987,0.908558,...,24.0,575.8,"[0.913, 0.916]","[0.058, 0.066]","[0.613, 0.715]",210,NaN,semantic_cohesion_lift_incl_noise 0.290 -> 0.2...,False,NaN
2,Sentence-T5 + Leiden,leiden,ADV_2_T5_LEIDEN__k_neighbors=20_resolution=2.0,"k_neighbors=20,resolution=2.0",31,1.000000,0.000000,0.898231,0.180002,0.898231,...,12.0,167.8,"[0.897, 0.900]","[0.042, 0.048]","[0.752, 0.828]",210,stability_ari 0.385 -> 0.562; paraphrase_kappa...,semantic_cohesion_lift_incl_noise 0.290 -> 0.180,False,NaN
3,Sentence-T5 + GMM,gmm,ADV_3_T5_GMM__n_components=140_cov=diag,"n_components=140,covariance_type=diag",140,1.000000,0.000000,0.918347,0.315454,0.918347,...,5.0,84.5,"[0.917, 0.920]","[0.061, 0.067]","[0.662, 0.742]",197,semantic_cohesion_lift_incl_noise 0.290 -> 0.315,pathology: unstable,False,NaN


CURRENT BASELINE: Sentence-T5 + K-Means | k=100
BEST CHALLENGER CONFIGS: {'bertopic': 'ADV_1_T5_BERTOPIC__nn=30_nc=5_mcs=10_ms=5', 'leiden': 'ADV_2_T5_LEIDEN__k_neighbors=20_resolution=2.0', 'gmm': 'ADV_3_T5_GMM__n_components=140_cov=diag'}
CHALLENGERS THAT QUALIFIED TO REPLACE K-MEANS: none
FINAL RECOMMENDATION: KEEP T5 + KMEANS
KEY REASON: no challenger shows a meaningful gain without an unacceptable cost
  bertopic: gains=[] costs=['semantic_cohesion_lift_incl_noise 0.290 -> 0.253', 'coverage 1.000 -> 0.728', 'pathology: unstable']
  leiden: gains=['stability_ari 0.385 -> 0.562', 'paraphrase_kappa 0.684 -> 0.782'] costs=['semantic_cohesion_lift_incl_noise 0.290 -> 0.180']
  gmm: gains=['semantic_cohesion_lift_incl_noise 0.290 -> 0.315'] costs=['pathology: unstable']


# Sprint 3 Advanced Challenger Report - does complexity earn its place?

Generated 2026-10-22 21:52 UTC by `sprint3_advanced_challengers.py`. All numbers are computed by the script; none are hand-entered. Representation frozen: `sentence-transformers/sentence-t5-base` (embeddings reused, not regenerated).

> **Known limitation.** The Sprint 3 input was section-filtered following a manually labelled task-purity audit. Post-filter audited task purity was 92.9%. Residual non-task contamination may therefore remain and is treated as a known limitation of the clustering experiment.

## Setup
- Baseline: `EXP_3_T5_KMEANS__k=100` (Sprint 3 overall winner (run_manifest.json chosen_configuration)), refit with identical parameters; reproduction check (`baseline/baseline_reproduction_check.json`): n_clusters diff 0.000000, semantic_cohesion diff 0.000000, semantic_cohesion_lift diff 0.000000, stability_ari diff 0.000000, silhouette diff 0.000000, provider_pair_kappa diff 0.000000, low_confidence_rate diff 0.000000; labels ARI vs recorded Sprint 3 membership 1.000
- Validation: {"input_sha256_matches_manifest_and_frozen": true, "input_rows": 3015, "embedding_shape": [3015, 768], "mapping_aligned": true, "embedding_meta_matches": true, "embeddings_normalised": true, "paired_wordings": 432, "pca_matches_sprint3": true}
- One evaluator for all methods (imported from the Sprint 3 script): semantic cohesion in the ORIGINAL T5 space with a size-preserving random-partition baseline (lift); silhouette in the shared Sprint 3 PCA space; the same 5 x 80% subsamples; the same provider nearest-neighbour test.
- Outlier-producing methods are compared on **coverage-adjusted** cohesion lift (outliers attached to their nearest centroid for the metric only) so that discarding hard statements is not rewarded; assigned-only values are also shown.
- Common low-confidence definition: primary-vs-secondary centroid margin below 0.0066 (P10 of the baseline's margins); each method's native confidence is reported separately.
- Replacement rule: a challenger replaces K-Means only with >= 1 gain beyond tolerance {"semantic_cohesion_lift_incl_noise": [0.02, "higher"], "stability_ari": [0.05, "higher"], "paraphrase_kappa": [0.05, "higher"], "provider_nn_gap_abs": [0.05, "lower"], "low_confidence_rate_common": [0.05, "lower"], "granularity_score": [0.25, "higher"]} and no cost beyond {"semantic_cohesion_lift_incl_noise": 0.02, "stability_ari": 0.05, "paraphrase_kappa": 0.05, "coverage": 0.1} or pathology flag.

## Comparison (best configuration per method)
| pipeline | parameters | n_clusters | coverage | semantic_cohesion_lift_incl_noise | kmeans_same_k_cohesion_lift | stability_ari | stability_ari_sd | paraphrase_consistency | paraphrase_kappa | provider_nn_gap_abs | silhouette | granularity_score | low_confidence_rate_common | secondary_membership_rate | pathology_flags |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| Sentence-T5 + K-Means | k=100 | 100 | 1.000 | 0.290 | 0.290 | 0.385 | 0.017 | 0.688 | 0.684 | 0.003 | 0.069 | 1.000 | 0.100 | 0.051 | unstable |
| Sentence-T5 + BERTopic | n_neighbors=30,n_components=5,min_dist=0.0,metric=cosine,min_cluster_size=10,min_samples=5 | 70 | 0.728 | 0.253 | 0.264 | 0.341 | 0.047 | 0.660 | 0.653 | 0.171 | 0.062 | 1.000 | 0.130 | 0.029 | unstable |
| Sentence-T5 + Leiden | k_neighbors=20,resolution=2.0 | 31 | 1.000 | 0.180 | 0.201 | 0.562 | 0.028 | 0.789 | 0.782 | 0.003 | 0.045 | 0.368 | 0.286 | 0.117 |  |
| Sentence-T5 + GMM | n_components=140,covariance_type=diag | 140 | 1.000 | 0.315 | 0.316 | 0.357 | 0.016 | 0.708 | 0.706 | 0.020 | 0.064 | 1.000 | 0.110 | 0.008 | unstable |

`kmeans_same_k_cohesion_lift` = K-Means refit at the challenger's topic count: it shows whether a cohesion difference is method quality or just granularity.

## Bootstrap 95% CIs (posting level, 100 resamples)
| method | metric | point_estimate | ci95_low | ci95_high |
|---|---|---|---|---|
| kmeans | semantic_cohesion | 0.914 | 0.913 | 0.915 |
| kmeans | semantic_cohesion_incl_noise | 0.914 | 0.913 | 0.915 |
| kmeans | silhouette | 0.069 | 0.066 | 0.071 |
| kmeans | paraphrase_consistency | 0.688 | 0.648 | 0.730 |
| bertopic | semantic_cohesion | 0.915 | 0.913 | 0.916 |
| bertopic | semantic_cohesion_incl_noise | 0.909 | 0.908 | 0.910 |
| bertopic | silhouette | 0.062 | 0.058 | 0.066 |
| bertopic | paraphrase_consistency | 0.660 | 0.613 | 0.715 |
| leiden | semantic_cohesion | 0.898 | 0.897 | 0.900 |
| leiden | semantic_cohesion_incl_noise | 0.898 | 0.897 | 0.900 |
| leiden | silhouette | 0.045 | 0.042 | 0.048 |
| leiden | paraphrase_consistency | 0.789 | 0.752 | 0.828 |
| gmm | semantic_cohesion | 0.918 | 0.917 | 0.920 |
| gmm | semantic_cohesion_incl_noise | 0.918 | 0.917 | 0.920 |
| gmm | silhouette | 0.064 | 0.061 | 0.067 |
| gmm | paraphrase_consistency | 0.708 | 0.662 | 0.742 |

## Answers
1. **Did BERTopic beat K-Means?** **No** - no gain beyond tolerance; costs: semantic_cohesion_lift_incl_noise 0.290 -> 0.253; coverage 1.000 -> 0.728; pathology: unstable.
2. **Did Leiden beat K-Means?** **No** - gains (stability_ari 0.385 -> 0.562; paraphrase_kappa 0.684 -> 0.782) come with unacceptable costs (semantic_cohesion_lift_incl_noise 0.290 -> 0.180).
3. **Did GMM beat K-Means?** **No** - gains (semantic_cohesion_lift_incl_noise 0.290 -> 0.315) come with unacceptable costs (pathology: unstable).
4. **Highest semantic cohesion:** Sentence-T5 + Leiden (coverage-adjusted lift 0.180; assigned-only highest: Sentence-T5 + Leiden). Compare with `kmeans_same_k_cohesion_lift` before attributing it to the method.
5. **Most stable:** Sentence-T5 + Leiden (subsample ARI 0.562).
6. **Best at preserving Gemini/Groq paraphrases:** Sentence-T5 + Leiden (chance-corrected kappa 0.782; raw rates favour coarse partitions, so kappa is the fair comparison).
7. **Handles ambiguous tasks best:** Sentence-T5 + Leiden (lowest common low-confidence rate 28.6%); secondary-membership rates: Sentence-T5 + K-Means 5.1%, Sentence-T5 + BERTopic 2.9%, Sentence-T5 + Leiden 11.7%, Sentence-T5 + GMM 0.8%.
8. **Most useful topic granularity:** Sentence-T5 + Leiden (granularity score 0.368, 31 topics, median size 93.000).
9. **Adds complexity without enough value:** Sentence-T5 + BERTopic (4 tuned hyper-parameters, extra deps: bertopic, umap-learn, hdbscan), Sentence-T5 + Leiden (2 tuned hyper-parameters, extra deps: python-igraph, leidenalg), Sentence-T5 + GMM (1 tuned hyper-parameters, extra deps: none (scikit-learn)).

## 10. FINAL RECOMMENDATION

**KEEP T5 + KMEANS**

Key reason: no challenger delivered a gain beyond the pre-declared tolerances without an unacceptable cost, so the simpler baseline is kept (complexity did not earn its place).

## Multi-topic membership rules (derived from each method's observed distributions)
- **Sentence-T5 + K-Means**: Sprint 3 centroid rule (secondary similarity >= P25 own-centroid similarity AND gap <= P10 gap) - thresholds {"secondary_min_similarity": 0.8988989293575287, "secondary_max_gap": 0.007227468490600586, "noise_post_hoc_min_similarity": 0.9141106605529785}
- **Sentence-T5 + BERTopic**: Sprint 3 centroid rule AND HDBSCAN soft membership of the secondary topic >= P75 of non-primary soft memberships; outliers assigned post hoc only by the Sprint 3 outlier rule - thresholds {"soft_membership_P75_non_primary": 0.007429887991235363, "secondary_min_similarity": 0.8974924236536026, "secondary_max_gap": 0.009571027755737305, "noise_post_hoc_min_similarity": 0.9151618480682373}
- **Sentence-T5 + Leiden**: Sprint 3 centroid rule AND edge-weight share to the secondary community >= P75 of positive non-own community shares - thresholds {"edge_share_P75": 0.08837477476797292, "secondary_min_similarity": 0.8827358782291412, "secondary_max_gap": 0.004312396049499512, "noise_post_hoc_min_similarity": 0.8988345265388489}
- **Sentence-T5 + GMM**: secondary = 2nd-highest posterior iff p2 >= max(P90 of p2, numerical floor) AND p1 - p2 <= P10 of (p1 - p2) - thresholds {"p2_P90": 0.001939270455879969, "numerical_floor": 0.05, "p2_threshold_used": 0.05, "gap_P10": 0.9957677045009885, "share_p2_below_floor": 0.9917081260364843}

## Per-method selection traces
**Sentence-T5 + BERTopic** -> `ADV_1_T5_BERTOPIC__nn=30_nc=5_mcs=10_ms=5`
- no non-pathological configuration - choosing among the least-flagged
- semantic_cohesion_lift_incl_noise: best=0.265 tol=0.02 -> kept ADV_1_T5_BERTOPIC__nn=15_nc=5_mcs=10_ms=5, ADV_1_T5_BERTOPIC__nn=15_nc=5_mcs=10_ms=10, ADV_1_T5_BERTOPIC__nn=15_nc=10_mcs=10_ms=5, ADV_1_T5_BERTOPIC__nn=15_nc=10_mcs=10_ms=10, ADV_1_T5_BERTOPIC__nn=30_nc=5_mcs=10_ms=5, ADV_1_T5_BERTOPIC__nn=30_nc=10_mcs=10_ms=5
- stability_ari: best=0.341 tol=0.05 -> kept ADV_1_T5_BERTOPIC__nn=30_nc=5_mcs=10_ms=5, ADV_1_T5_BERTOPIC__nn=30_nc=10_mcs=10_ms=5
- granularity_score: best=1.000 tol=0.0 -> kept ADV_1_T5_BERTOPIC__nn=30_nc=5_mcs=10_ms=5, ADV_1_T5_BERTOPIC__nn=30_nc=10_mcs=10_ms=5
- robustness: best=0.653 tol=0.05 -> kept ADV_1_T5_BERTOPIC__nn=30_nc=5_mcs=10_ms=5
- decided by robustness

**Sentence-T5 + Leiden** -> `ADV_2_T5_LEIDEN__k_neighbors=20_resolution=2.0`
- semantic_cohesion_lift_incl_noise: best=0.180 tol=0.02 -> kept ADV_2_T5_LEIDEN__k_neighbors=20_resolution=2.0, ADV_2_T5_LEIDEN__k_neighbors=30_resolution=2.0
- stability_ari: best=0.564 tol=0.05 -> kept ADV_2_T5_LEIDEN__k_neighbors=20_resolution=2.0, ADV_2_T5_LEIDEN__k_neighbors=30_resolution=2.0
- granularity_score: best=0.368 tol=0.0 -> kept ADV_2_T5_LEIDEN__k_neighbors=20_resolution=2.0
- decided by granularity_score

**Sentence-T5 + GMM** -> `ADV_3_T5_GMM__n_components=140_cov=diag`
- no non-pathological configuration - choosing among the least-flagged
- semantic_cohesion_lift_incl_noise: best=0.315 tol=0.02 -> kept ADV_3_T5_GMM__n_components=120_cov=diag, ADV_3_T5_GMM__n_components=140_cov=diag
- stability_ari: best=0.360 tol=0.05 -> kept ADV_3_T5_GMM__n_components=120_cov=diag, ADV_3_T5_GMM__n_components=140_cov=diag
- granularity_score: best=1.000 tol=0.0 -> kept ADV_3_T5_GMM__n_components=120_cov=diag, ADV_3_T5_GMM__n_components=140_cov=diag
- robustness: best=0.706 tol=0.05 -> kept ADV_3_T5_GMM__n_components=120_cov=diag, ADV_3_T5_GMM__n_components=140_cov=diag
- silhouette: best=0.065 tol=0.01 -> kept ADV_3_T5_GMM__n_components=120_cov=diag, ADV_3_T5_GMM__n_components=140_cov=diag
- decided by tie_break

## Limitations
- The Sprint 3 input was section-filtered following a manually labelled task-purity audit. Post-filter audited task purity was 92.9%. Residual non-task contamination may therefore remain and is treated as a known limitation of the clustering experiment.
- No ground-truth topics: all metrics are internal or robustness measures, not accuracy.
- BERTopic keywords (`bertopic/bertopic_keywords_all_configs.csv`) are for inspection only, not taxonomy names.
- GMM BIC/AIC are supporting evidence only. Bootstrap CIs are conditional on the fitted partition.
- Paraphrase pairs come from Sprint 2 cross-model agreement and skew towards easier paraphrases.

Runtime: {"gmm_seconds": 84.5, "leiden_seconds": 167.8, "bertopic_seconds": 575.8, "total_seconds": 924.1}